# Day 5: Neurons — The Biological Unit

**Learning Objective**: Implement a `Neuron` class that applies weights, bias, and activation to inputs.

A neuron is the fundamental computational unit of neural networks:
```
inputs → [weighted sum] → [add bias] → [activation] → output
```

In [ ]:
import math
import random
from graphviz import Digraph

## Theory: The Neuron Equation

$$y = \phi\left(\sum_{i=1}^{n} w_i \cdot x_i + b\right)$$

Where:
- $x_i$ = inputs
- $w_i$ = weights (learnable)
- $b$ = bias (learnable)
- $\phi$ = activation function (tanh)

### Why Activation Functions?

Without non-linearity, stacked layers collapse to one linear transformation.
Non-linearity lets networks learn complex patterns.

### Tanh Properties
- Output range: (-1, 1)
- Centered at 0
- Derivative: $1 - \tanh^2(x)$

## Value Class (from Day 4)

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers"
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        """Tanh activation with derivative 1 - tanh²(x)."""
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return Value(other) - self

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self, other):
        return Value(other) * self**-1

## Visualization Helper

In [ ]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

## The Neuron Class

A neuron has:
- **Weights**: One weight per input
- **Bias**: Shifts the activation threshold
- **Activation**: Non-linear function (tanh)

In [ ]:
class Neuron:
    def __init__(self, nin):
        """
        nin: number of inputs to this neuron
        """
        # Weight initialization: random values between -1 and 1
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        # Bias initialization
        self.b = Value(0)
    
    def __call__(self, x):
        """
        Forward pass: weighted sum + bias + activation
        """
        # Weighted sum: w·x + b
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        # Activation
        out = act.tanh()
        return out
    
    def parameters(self):
        """Return all learnable parameters."""
        return self.w + [self.b]

## Test 1: Basic Neuron Forward Pass

Create a neuron with 3 inputs and verify it produces an output in range (-1, 1).

In [ ]:
# Create a neuron with 3 inputs
n = Neuron(3)

# Show weights and bias
print("Weights:", [w.data for w in n.w])
print("Bias:", n.b.data)
print(f"Number of parameters: {len(n.parameters())}")

# Forward pass with input
x = [Value(2.0), Value(3.0), Value(-1.0)]
out = n(x)

print(f"\nOutput: {out.data:.4f}")
assert -1 < out.data < 1, "Output should be in (-1, 1) range due to tanh"
print("✅ Neuron produces valid output!")

## Test 2: Backpropagation Through a Neuron

Verify that gradients flow correctly through the neuron.

In [ ]:
# Create fresh neuron
n = Neuron(2)
x = [Value(1.0, label='x1'), Value(2.0, label='x2')]

# Forward pass
out = n(x)
out.label = 'out'
print(f"Output: {out.data:.4f}")

# Backward pass
out.backward()

# Check gradients
print("\nGradients:")
for i, w in enumerate(n.w):
    print(f"  w[{i}].grad = {w.grad:.4f}")
print(f"  b.grad = {n.b.grad:.4f}")

# Verify gradients are non-zero
assert any(w.grad != 0 for w in n.w), "Weight gradients should be non-zero"
assert n.b.grad != 0, "Bias gradient should be non-zero"
print("\n✅ Gradients flow correctly through neuron!")

## Test 3: Trace Neuron Computation Step by Step

In [ ]:
def trace_neuron(neuron, inputs):
    """Print step-by-step computation."""
    print("=" * 40)
    print("Neuron Computation Trace")
    print("=" * 40)
    print(f"Inputs: {[x.data for x in inputs]}")
    print(f"Weights: {[w.data for w in neuron.w]}")
    print(f"Bias: {neuron.b.data:.4f}")
    
    # Weighted sum
    weighted = [w.data * x.data for w, x in zip(neuron.w, inputs)]
    print(f"\nWeighted inputs: {weighted}")
    print(f"Sum of weighted inputs: {sum(weighted):.4f}")
    print(f"Sum + bias: {sum(weighted) + neuron.b.data:.4f}")
    
    # Output
    out = neuron(inputs)
    print(f"After tanh: {out.data:.4f}")
    print("=" * 40)
    return out

# Trace a neuron
n = Neuron(2)
x = [Value(1.0), Value(-2.0)]
out = trace_neuron(n, x)

## Test 4: Visualize Neuron Computation Graph

In [ ]:
# Create labeled neuron
n = Neuron(2)
n.w[0].label = 'w1'
n.w[1].label = 'w2'
n.b.label = 'b'

x1 = Value(2.0, label='x1')
x2 = Value(3.0, label='x2')

out = n([x1, x2])
out.label = 'out'

out.backward()

# Visualize
draw_dot(out)

## Worked Example

A neuron with 2 inputs, weights [0.5, -0.3], bias 0.1:

```
Forward:
  x = [2.0, 3.0]
  weighted_sum = 0.5 * 2.0 + (-0.3) * 3.0 + 0.1
               = 1.0 - 0.9 + 0.1
               = 0.2
  output = tanh(0.2) ≈ 0.197

Backward (assuming out.grad = 1.0):
  d(tanh)/d(input) = 1 - tanh²(0.2) ≈ 0.961
  
  d/dw₀ = x₀ * 0.961 = 2.0 * 0.961 = 1.922
  d/dw₁ = x₁ * 0.961 = 3.0 * 0.961 = 2.883
  d/db = 1 * 0.961 = 0.961
```

In [ ]:
# Verify worked example
class FixedNeuron:
    """Neuron with fixed weights for testing."""
    def __init__(self, weights, bias):
        self.w = [Value(w) for w in weights]
        self.b = Value(bias)
    
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

n = FixedNeuron([0.5, -0.3], 0.1)
x = [Value(2.0), Value(3.0)]
out = n(x)

print(f"Output: {out.data:.4f} (expected ~0.197)")

out.backward()

print(f"\nGradients:")
print(f"  w[0].grad = {n.w[0].grad:.4f} (expected ~1.922)")
print(f"  w[1].grad = {n.w[1].grad:.4f} (expected ~2.883)")
print(f"  b.grad = {n.b.grad:.4f} (expected ~0.961)")

# Verify
assert abs(out.data - 0.197) < 0.01
assert abs(n.w[0].grad - 1.922) < 0.01
print("\n🎉 Worked example verified!")

## Summary

Today we implemented:

1. **`Neuron` class** with:
   - Random weight initialization
   - Bias term
   - tanh activation
   - `parameters()` method for accessing learnable values

2. **Verified** that:
   - Output is in (-1, 1) range due to tanh
   - Gradients flow correctly through the neuron
   - Computation matches hand-calculated values

**Key insight**: A neuron is just a parameterized function that computes a weighted sum, adds a bias, and applies a non-linear activation. The magic is that gradients flow backward through this computation!

---

*Previous: [Day 4 — More Operations](./day_04_more_operations.ipynb)*  
*Next: Day 6 — Layers*